# PSSN Week 2 — Expand and validate CoT traces

Run cells in order in a Colab GPU runtime. This notebook runs the fixed Qwen3-4B GSM8K CoT baseline on 700 deterministically sampled, previously unattempted training questions with a 1,024-token cap. The run is resumable across Colab sessions and processes at most 75 new questions each time the generation cell is run. It preserves every raw attempt, including exceptions and incomplete generations, and writes a separate segmentation-ready file containing only completed, parseable, correct traces. The planning target is about 300 usable traces; actual yield may be lower or higher.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from google.colab import drive
drive.mount("/content/drive")
print("Repository:", Path.cwd())


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import random
import torch

from src.utils.environment import collect_environment_info
from src.data.gsm8k import load_gsm8k
from src.models.qwen import GenerationSettings, QwenGenerator
from src.baseline.cot import BASELINE_PROMPT_TEMPLATE
from src.utils import ExperimentLogger

SEED = 44
NUMBER_OF_PROBLEMS = 700
MAX_PROBLEMS_PER_SESSION = 75
MAX_NEW_TOKENS = 1024
TARGET_VALID_TRACES = 300
MODEL_ID = "Qwen/Qwen3-4B"
assert 0 < MAX_PROBLEMS_PER_SESSION <= NUMBER_OF_PROBLEMS

environment = collect_environment_info()
assert environment["cuda_available"], "Select a Colab GPU runtime."
if environment["gpus"][0]["compute_capability"] == "7.5":
    assert environment["selected_dtype"] == "float16", "T4 must use FP16. Sync updated src/ and restart the runtime."
print(json.dumps(environment, indent=2))

gsm8k = load_gsm8k(seed=SEED)
train = gsm8k["train"]
assert 0 < NUMBER_OF_PROBLEMS <= len(train)

# Use a stable run folder so the same sample and append-only log resume in later sessions.
output_dir = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline/w2-001-700-attempts-seed44")
output_dir.mkdir(parents=True, exist_ok=True)
run_dataset_path = output_dir / "run_dataset"
raw_path = output_dir / "raw_attempts.jsonl"

if run_dataset_path.exists():
    from datasets import Dataset
    examples = Dataset.load_from_disk(str(run_dataset_path))
    assert len(examples) == NUMBER_OF_PROBLEMS, "Saved run dataset does not match the requested 700-question run."
    print("Resuming saved sample; questions:", len(examples))
else:
    # Avoid repeating questions attempted by earlier expanded-baseline runs.
    attempted_elsewhere = set()
    previous_runs_dir = output_dir.parent
    for previous_raw in previous_runs_dir.glob("*/raw_attempts.jsonl"):
        if previous_raw.parent == output_dir:
            continue
        for line in previous_raw.read_text(encoding="utf-8").splitlines():
            if line.strip():
                attempted_elsewhere.add(str(json.loads(line).get("problem_id")))
    available = [index for index, item in enumerate(train) if item["id"] not in attempted_elsewhere]
    assert len(available) >= NUMBER_OF_PROBLEMS, f"Only {len(available)} unattempted GSM8K training rows remain."
    selected_indices = sorted(random.Random(SEED).sample(available, NUMBER_OF_PROBLEMS))
    examples = train.select(selected_indices)
    examples.save_to_disk(str(run_dataset_path))
assert len(examples) == NUMBER_OF_PROBLEMS
settings = GenerationSettings(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, seed=SEED)
generator = QwenGenerator.from_pretrained(model_id=MODEL_ID, settings=settings)
print("Loaded precision:", generator.precision)

existing_configuration_path = output_dir / "run_configuration.json"
existing_configuration = json.loads(existing_configuration_path.read_text(encoding="utf-8")) if existing_configuration_path.exists() else {}
logger = ExperimentLogger(
    output_dir / "raw_attempts.jsonl",
    experiment_name="qwen3-4b-gsm8k-expanded-cot-week2",
    run_id=existing_configuration.get("run_id"),
    timestamp=datetime.fromisoformat(existing_configuration["captured_at_utc"]) if existing_configuration.get("captured_at_utc") else None,
    run_metadata={
        "dataset": "openai/gsm8k", "subset_source": "train",
        "number_of_problems": NUMBER_OF_PROBLEMS, "max_problems_per_session": MAX_PROBLEMS_PER_SESSION,
        "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
    },
)

def model_revision(component):
    return getattr(getattr(component, "config", None), "_commit_hash", None) or getattr(component, "init_kwargs", {}).get("_commit_hash")

configuration = {
    "run_id": logger.run_id,
    "captured_at_utc": logger.timestamp_utc,
    "repository_url": REPOSITORY_URL,
    "repository_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip(),
    "environment": environment,
    "packages": {name: version(name) for name in ["torch", "transformers", "datasets", "accelerate", "safetensors"]},
    "model_id": MODEL_ID,
    "model_revision": model_revision(generator.model),
    "tokenizer_revision": model_revision(generator.tokenizer),
    "model_metadata": {
        "model_class": type(generator.model).__name__,
        "precision": generator.precision,
        "device": str(generator.model.device),
        "device_map": getattr(generator.model, "hf_device_map", None),
        "quantized": False,
    },
    "generation": settings.to_dict(),
    "thinking_enabled": True,
    "prompt_template": BASELINE_PROMPT_TEMPLATE,
    "training_rows_available": len(train),
    "number_of_problems": NUMBER_OF_PROBLEMS,
    "max_problems_per_session": MAX_PROBLEMS_PER_SESSION,
    "seed": SEED,
    "dataset_fingerprint": examples._fingerprint,
    "selected_example_ids": list(examples["id"]),
    "planning_target_usable_traces": TARGET_VALID_TRACES,
    "segmentation_policy": "Only complete, parseable, correct traces are segmentation eligible",
    "incomplete_policy": "Preserve in raw_attempts.jsonl; exclude from evaluation and segmentation",
}
if not existing_configuration_path.exists():
    existing_configuration_path.write_text(json.dumps(configuration, indent=2) + "\n", encoding="utf-8")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
    (output_dir / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
elif existing_configuration.get("selected_example_ids") != list(examples["id"]):
    raise RuntimeError("Saved run configuration and persisted question sample do not match; preserve this folder and investigate before proceeding.")
print("Run folder:", output_dir)
print("Saved exact run dataset:", run_dataset_path)


## Generate and preserve all attempts

This cell appends each result immediately. If Colab interrupts the run, rerun this cell in the same runtime: it reads saved problem IDs and skips already logged questions.

In [ ]:
from src.baseline.cot import generate_cot_record
from src.evaluation.accuracy import evaluate_record

raw_path = output_dir / "raw_attempts.jsonl"
already_saved = set()
if raw_path.exists():
    for line in raw_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            already_saved.add(json.loads(line)["problem_id"])

processed_this_session = 0
for index, example in enumerate(examples, start=1):
    if example["id"] in already_saved:
        continue
    if processed_this_session >= MAX_PROBLEMS_PER_SESSION:
        break
    try:
        record = generate_cot_record(example, generator)
        evaluation = evaluate_record(record)
        record.update({
            "predicted_answer": evaluation["predicted_answer"],
            "normalized_reference_answer": evaluation["normalized_reference_answer"],
            "normalized_predicted_answer": evaluation["normalized_predicted_answer"],
            "is_correct": evaluation["is_correct"],
            "evaluation_status": evaluation["evaluation_status"],
        })
    except Exception as exc:
        record = {
            "problem_id": example["id"],
            "question": example["question"],
            "reference_answer": str(example["reference_answer"]),
            "reasoning_trace": "",
            "predicted_answer": None,
            "is_correct": None,
            "evaluation_status": "generation_error",
            "input_tokens": 0,
            "output_tokens": 0,
            "finish_reason": "error",
            "generation_config": settings.to_dict(),
            "model_metadata": {"model_id": MODEL_ID, "precision": generator.precision},
            "generation_error": f"{type(exc).__name__}: {exc}",
        }
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = logger.log(record)
    already_saved.add(example["id"])
    processed_this_session += 1
    status = saved.get("finish_reason", "unknown")
    print(f"{index}/{NUMBER_OF_PROBLEMS}: {example['id']} | {saved['output_tokens']} tokens | stop={status}")

print("New attempts this session:", processed_this_session)
print("Raw attempts saved:", len(already_saved), "of", NUMBER_OF_PROBLEMS)
print("Rerun this cell in a later session until all 700 attempts are saved.")


## Validate and create the segmentation input

All attempts remain in `raw_attempts.jsonl`. Completed but incorrect traces are retained and labeled in the validation log. Only completed traces with an extracted answer marked correct go into `segmentation_ready_traces.jsonl`.

In [ ]:
from src.data.trace_validation import validate_traces, summarize_trace_validation

raw_records = [
    json.loads(line)
    for line in raw_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
validation_results = validate_traces(raw_records)
summary = summarize_trace_validation(
    raw_records,
    validation_results,
    target_valid_traces=TARGET_VALID_TRACES,
)

validation_path = output_dir / "trace_validation.jsonl"
segmentation_path = output_dir / "segmentation_ready_traces.jsonl"
invalid_path = output_dir / "invalid_or_incomplete_traces.jsonl"
validation_path.write_text(
    "".join(json.dumps(item, ensure_ascii=False) + "\n" for item in validation_results),
    encoding="utf-8",
)
segmentation_records = [
    record for record, check in zip(raw_records, validation_results)
    if check["segmentation_eligible"]
]
invalid_records = [
    {**record, "validation": check}
    for record, check in zip(raw_records, validation_results)
    if not check["trace_valid"]
]
segmentation_path.write_text(
    "".join(json.dumps(item, ensure_ascii=False) + "\n" for item in segmentation_records),
    encoding="utf-8",
)
invalid_path.write_text(
    "".join(json.dumps(item, ensure_ascii=False) + "\n" for item in invalid_records),
    encoding="utf-8",
)
summary["run_id"] = logger.run_id
summary["source_raw_attempts"] = str(raw_path)
summary["validation_log"] = str(validation_path)
summary["segmentation_ready_file"] = str(segmentation_path)
summary["invalid_or_incomplete_file"] = str(invalid_path)
(output_dir / "trace_validation_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
if not summary["target_valid_traces_met"]:
    print(f"Planning target not met: {TARGET_VALID_TRACES} valid completed traces; this is an estimate, not a guaranteed yield.")
if not summary["target_segmentation_traces_met"]:
    print(f"Fewer than {TARGET_VALID_TRACES} complete correct traces are ready for successful-trace segmentation.")
